# PDF Submission Delta Analysis

Analyze extracted PDF fields and compute deltas between successive submissions for sample companies.
This notebook:
1. Loads extracted PDF submission data
2. Groups submissions by company and time period
3. Computes deltas (changes) in revenue, employees, controls, and policies
4. Identifies key patterns and generates sample recommendations
5. Prepares data for dashboard initialization

## 1. Setup & Imports

In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import sys

# Setup paths
os.chdir('..')  # Move to project root
sys.path.insert(0, '/c/PROJECT')

# Import custom modules
from src.business.models import Delta, RiskScore, Submission, Customer
from src.business.scoring import ScoringRules, Recommender

print("[SETUP] Environment configured")
print(f"Working directory: {Path.cwd()}")
print(f"Python version: {sys.version.split()[0]}")

# Display options
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.max_rows', 100)

[SETUP] Environment configured
Working directory: c:\PROJECT
Python version: 3.12.13


## 2. Load Extracted PDF Data

In [3]:
# Load extracted PDF fields
pdf_extracted = pd.read_csv('data/parsed/pdf_extracted_fields.csv')

# Load Excel submission baseline
all_submissions = pd.read_csv('data/parsed/all_submissions.csv')
repeat_customers = pd.read_csv('data/parsed/repeat_customers.csv')

print(f"[LOAD] PDF extracted fields: {len(pdf_extracted)} rows, {len(pdf_extracted.columns)} cols")
print(f"[LOAD] All submissions (Excel): {len(all_submissions)} rows")
print(f"[LOAD] Repeat customers: {len(repeat_customers)} rows")

# Display sample PDF extraction
print(f"\n📋 PDF Extraction Sample:")
print(pdf_extracted[['company_name', 'pdf_file', 'extraction_success', 'revenue_millions', 'employee_count']].head())

print(f"\n📋 Excel Submission Sample (columns: {list(all_submissions.columns)[:6]}):")
print(all_submissions.iloc[:, :6].head())

[LOAD] PDF extracted fields: 162 rows, 26 cols
[LOAD] All submissions (Excel): 46318 rows
[LOAD] Repeat customers: 9078 rows

📋 PDF Extraction Sample:
           company_name                                              pdf_file  \
0  Additional Documents                      Global CI Cyber UW Guideline.pdf   
1  Additional Documents                ZNA PL&C MM Cyber Playbook 2.01.26.pdf   
2  Additional Documents                     Zurich Cyber Brochure - Oct18.pdf   
3             Company 1  2022 04 CYSL TECH EO App - Cyber and Tech EO App.pdf   
4             Company 1               2023 04 CYSL App - Cyber App_Signed.pdf   

   extraction_success  revenue_millions  employee_count  
0                True               NaN             NaN  
1                True            100.00             NaN  
2                True              3.86             NaN  
3                True               NaN             NaN  
4                True               NaN             NaN  

📋 Excel Submi

## 3. Define Delta Computation Functions

In [4]:
def compute_simple_delta(baseline, current):
    """
    Compute percentage change between baseline and current values.
    Returns (pct_change, absolute_change) or (None, None) if either is null.
    """
    if pd.isna(baseline) or pd.isna(current) or baseline == 0:
        return None, None
    
    pct_change = ((current - baseline) / abs(baseline)) * 100
    abs_change = current - baseline
    return pct_change, abs_change

def compute_control_delta(baseline_controls, current_controls):
    """
    Compare control flags between baseline and current.
    Returns dict with added, removed, and consistent controls.
    """
    baseline_set = set([col for col, val in baseline_controls.items() if val])
    current_set = set([col for col, val in current_controls.items() if val])
    
    added = current_set - baseline_set
    removed = baseline_set - current_set
    consistent = baseline_set & current_set
    
    return {
        'added': list(added),
        'removed': list(removed),
        'consistent': list(consistent),
        'count_added': len(added),
        'count_removed': len(removed),
    }

def build_sample_deltas(max_companies=5):
    """
    Build sample deltas for companies with 2+ PDF submissions.
    Uses PDF extraction data and Excel baseline.
    """
    deltas = []
    
    companies_with_multiple_pdfs = pdf_extracted.groupby('company_name').size()
    companies_with_multiple_pdfs = companies_with_multiple_pdfs[companies_with_multiple_pdfs >= 2].index[:max_companies]
    
    for company in companies_with_multiple_pdfs:
        company_pdfs = pdf_extracted[pdf_extracted['company_name'] == company].sort_values('pdf_file')
        
        if len(company_pdfs) < 2:
            continue
        
        # Use first as baseline, last as current
        baseline = company_pdfs.iloc[0]
        current = company_pdfs.iloc[-1]
        
        # Extract control flags
        control_cols = [c for c in company_pdfs.columns if c.startswith('control_')]
        baseline_controls = {col: baseline[col] for col in control_cols}
        current_controls = {col: current[col] for col in control_cols}
        
        # Compute deltas
        rev_pct, rev_abs = compute_simple_delta(baseline['revenue_millions'], current['revenue_millions'])
        emp_pct, emp_abs = compute_simple_delta(baseline['employee_count'], current['employee_count'])
        control_delta = compute_control_delta(baseline_controls, current_controls)
        
        delta_record = {
            'company_name': company,
            'baseline_pdf': baseline['pdf_file'],
            'current_pdf': current['pdf_file'],
            'revenue_pct_change': rev_pct,
            'employee_pct_change': emp_pct,
            'controls_added': control_delta['count_added'],
            'controls_removed': control_delta['count_removed'],
            'baseline_premium': baseline['premium_usd'],
            'current_premium': current['premium_usd'],
            'baseline_date': baseline['policy_effective_date'],
            'current_date': current['policy_effective_date'],
        }
        
        deltas.append(delta_record)
    
    return pd.DataFrame(deltas)

print("[DEFINE] Delta computation functions ready")

[DEFINE] Delta computation functions ready


## 4. Compute Sample Deltas & Score

In [ ]:
# Build sample deltas from PDF data
sample_deltas = build_sample_deltas(max_companies=5)

print(f"\n✅ [ANALYSIS] Built {len(sample_deltas)} sample deltas from PDF submissions")
print(f"\n📊 Delta Summary:")
print(sample_deltas.to_string())

# Compute risk scores for each delta (simplified without full Delta model)
print(f"\n🎯 RISK SCORING:")
recommendations = []

for idx, row in sample_deltas.iterrows():
    # Compute score directly using ScoringRules logic
    # Create a simple dict representation for scoring
    revenue_pct = row['revenue_pct_change'] if pd.notna(row['revenue_pct_change']) else 0
    emp_pct = row['employee_pct_change'] if pd.notna(row['employee_pct_change']) else 0
    controls_added = int(row['controls_added']) if pd.notna(row['controls_added']) else 0
    controls_removed = int(row['controls_removed']) if pd.notna(row['controls_removed']) else 0
    
    # Apply scoring logic (simplified)
    # Revenue risk: 0% = 0, ±20% = 10, ±50% = 25
    rev_points = 0
    if revenue_pct != 0:
        if abs(revenue_pct) >= 50:
            rev_points = 25 * (abs(revenue_pct) / 50)
        elif abs(revenue_pct) >= 20:
            rev_points = 10 * (abs(revenue_pct) / 20)
        elif abs(revenue_pct) >= 5:
            rev_points = 5 * (abs(revenue_pct) / 5)
    
    # Control risk: degraded/removed = +10 each, improved = -5 each
    control_points = (controls_removed * 10) - (controls_added * 5)
    
    # Base score: start at 50, add/subtract points, clamp 0-100
    base_score = max(0, min(100, 50 + rev_points + control_points))
    
    # Determine recommendation
    if base_score < 35:
        recommendation = "FAST_TRACK"
    elif base_score < 65:
        recommendation = "STANDARD_UW"
    else:
        recommendation = "FRESH_UW"
    
    rec_record = {
        'company_name': row['company_name'],
        'risk_score': base_score,
        'recommendation': recommendation,
        'confidence': 0.75,
        'reasoning': f"Revenue change: {revenue_pct:.0f}%, Controls changed: -{controls_removed}+{controls_added}",
    }
    
    recommendations.append(rec_record)
    print(f"  {row['company_name']}: Score={base_score:.0f}, Rec={recommendation}")

# Convert to DataFrame
recs_df = pd.DataFrame(recommendations)

# Save for dashboard
recs_df.to_csv('data/parsed/sample_recommendations.csv', index=False)
print(f"\n💾 [DONE] Saved {len(recs_df)} sample recommendations to data/parsed/sample_recommendations.csv")

print(f"\n📋 Final Recommendation List:")
print(recs_df.to_string())


✅ [ANALYSIS] Built 5 sample deltas from PDF submissions

📊 Delta Summary:
           company_name                                          baseline_pdf                                                    current_pdf revenue_pct_change employee_pct_change  controls_added  controls_removed  baseline_premium  current_premium  baseline_date  current_date
0  Additional Documents                      Global CI Cyber UW Guideline.pdf                              Zurich Cyber Brochure - Oct18.pdf               None                None               0                 3               NaN              NaN            NaN           NaN
1             Company 1  2022 04 CYSL TECH EO App - Cyber and Tech EO App.pdf    Signed_2021_04_CYSL_TECH_EO_App_-_Cyber_and_Tech_EO_App.pdf               None                None               0                 0               NaN              NaN            NaN           NaN
2            Company 10          2023 05 CYSL EXP - Completed Application.pdf  Ransomware A

ValidationError: 5 validation errors for Delta
from_submission_id
  Field required [type=missing, input_value={'submission_id': 'pdf_0'...0, 'anomaly_score': 0.0}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
to_submission_id
  Field required [type=missing, input_value={'submission_id': 'pdf_0'...0, 'anomaly_score': 0.0}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
months_between
  Field required [type=missing, input_value={'submission_id': 'pdf_0'...0, 'anomaly_score': 0.0}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
controls_added
  Input should be a valid list [type=list_type, input_value=0, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/list_type
controls_removed
  Input should be a valid list [type=list_type, input_value=3, input_type=int]
    For further information visit https://errors.pydantic

## 5. Visualize Results & Export Summary

In [ ]:
# Summary table for visualization
print(f"\n" + "="*80)
print(f"ZURICH HACKATHON - PDF DELTA ANALYSIS SUMMARY")
print(f"="*80)

print(f"\n✅ Extraction Status:")
print(f"  PDFs successfully parsed: {pdf_extracted['extraction_success'].sum()}/{len(pdf_extracted)}")
print(f"  Companies with multiple submissions: {(pdf_extracted.groupby('company_name').size() >= 2).sum()}")

print(f"\n📊 Delta Statistics (n={len(sample_deltas)}):")
if len(sample_deltas) > 0:
    print(f"  Avg revenue change: {sample_deltas['revenue_pct_change'].mean():.1f}%")
    print(f"  Avg employee change: {sample_deltas['employee_pct_change'].mean():.1f}%")
    print(f"  Avg controls added: {sample_deltas['controls_added'].mean():.1f}")

print(f"\n🎯 Recommendation Distribution:")
print(recs_df['recommendation'].value_counts().to_string())

print(f"\n📁 Data Files Generated:")
print(f"  ✓ data/parsed/pdf_extracted_fields.csv - {len(pdf_extracted)} extraction records")
print(f"  ✓ data/parsed/sample_deltas.csv - {len(sample_deltas)} delta records")
print(f"  ✓ data/parsed/sample_recommendations.csv - {len(recs_df)} recommendation records")

# Save deltas too
sample_deltas.to_csv('data/parsed/sample_deltas.csv', index=False)

print(f"\n✅ [COMPLETE] PDF analysis notebook finished successfully")
print(f"Next: Build Knowledge Graph (notebooks/03_kg_construction.ipynb)")